# Oasis Infobyte Data Analytics Internship
## Project 2: House Price Prediction using Linear Regression
**Author:** Jasmin Jamadar  
**Role:** Data Analyst Intern  
**Project Objective:** Build a Multiple Linear Regression model to predict house prices based on structural and locational features and extract actionable insights for the real estate market.

---



### Project Outline
1. **Environment Setup & Libraries** - Load core libraries (pandas, numpy, scikit-learn, etc.).
2. **Dataset Loading & Initial Inspection** - Ingesting and examining the shape, head, columns, and properties.
3. **Data Cleaning & Outlier Treatment** - Checking duplicates, missing values, and applying IQR filtering to outliers.
4. **Exploratory Data Analysis (EDA)** - Statistical summaries, price distributions, and feature relationships.
5. **Feature Encoding** - Mapping binary features and dummy-encoding categorical features.
6. **Correlation Analysis** - Correlation matrix and heatmap visualization.
7. **Train-Test Split & Scaling** - 80/20 data split and MinMaxScaler alignment.
8. **Model Building & Training** - Building a Multiple Linear Regression model.
9. **Performance Evaluation** - Estimating model accuracy via MAE, MSE, RMSE, and $R^2$ Score.
10. **Model Diagnostics** - Checking residuals distribution, homoscedasticity, and actual vs predicted trends.
11. **Key Insights & Business Recommendations** - Summarizing the most influential pricing factors.



### 1. Environment Setup & Library Imports
We begin by loading standard data science packages for preprocessing, modeling, and visualization.



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set style for professional plots
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
})
print("Environment setup complete.")


### 2. Dataset Loading & Initial Inspection
We load the dataset `Housing.csv` from the `Dataset` folder and examine its structure.



In [ ]:
# Define dataset path
dataset_path = os.path.join("..", "Dataset", "Housing.csv")

# Load data
df = pd.read_csv(dataset_path)
print(f"Dataset successfully loaded. Shape: {df.shape}")
print("\n--- Columns in Dataset ---")
print(df.columns.tolist())
print("\n--- Dataset Head ---")
print(df.head())
print("\n--- Dataset Info ---")
df.info()


### 3. Data Cleaning & Outlier Treatment
We check for missing values and duplicates, and perform outlier detection.
Linear regression is highly sensitive to outliers, particularly in continuous columns like `price` and `area`. We will filter out values beyond the standard $1.5 \times IQR$ range.



In [ ]:
# Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Check duplicates
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

# Plot initial boxplots for continuous variables
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df['price'], color='#4f46e5', ax=axs[0])
axs[0].set_title('Price Outliers Check (Raw)')
sns.boxplot(y=df['area'], color='#10b981', ax=axs[1])
axs[1].set_title('Area Outliers Check (Raw)')
plt.tight_layout()
plt.show()


In [ ]:
# Outlier removal using IQR method for 'price'
Q1_price = df['price'].quantile(0.25)
Q3_price = df['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
price_lower = Q1_price - 1.5 * IQR_price
price_upper = Q3_price + 1.5 * IQR_price

# Outlier removal using IQR method for 'area'
Q1_area = df['area'].quantile(0.25)
Q3_area = df['area'].quantile(0.75)
IQR_area = Q3_area - Q1_area
area_lower = Q1_area - 1.5 * IQR_area
area_upper = Q3_area + 1.5 * IQR_area

# Filter the dataframe
df_clean = df[
    (df['price'] >= price_lower) & (df['price'] <= price_upper) &
    (df['area'] >= area_lower) & (df['area'] <= area_upper)
].reset_index(drop=True)

print(f"Original dataset records: {len(df)}")
print(f"Cleaned dataset records: {len(df_clean)}")
print(f"Dropped {len(df) - len(df_clean)} outlier records.")

# Replot clean boxplots
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df_clean['price'], color='#818cf8', ax=axs[0])
axs[0].set_title('Price Boxplot (Outliers Removed)')
sns.boxplot(y=df_clean['area'], color='#34d399', ax=axs[1])
axs[1].set_title('Area Boxplot (Outliers Removed)')
plt.tight_layout()
plt.show()


### 4. Exploratory Data Analysis (EDA)
We conduct price distribution analysis, category relationships, and statistical summaries.



In [ ]:
# Statistical Summary
print("--- Statistical Summary of Cleaned Data ---")
print(df_clean.describe())

# Price Distribution Plot
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['price'], kde=True, color='#4f46e5')
plt.title('House Price Distribution (Cleaned Data)')
plt.xlabel('Price')
plt.ylabel('Count')
os.makedirs(os.path.join("..", "Visualizations"), exist_ok=True)
plt.savefig(os.path.join("..", "Visualizations", "house_price_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Relationships of categories with Price
plt.figure(figsize=(15, 10))
plt.subplot(2, 3, 1)
sns.boxplot(x='mainroad', y='price', data=df_clean, palette='muted', hue='mainroad', legend=False)
plt.subplot(2, 3, 2)
sns.boxplot(x='guestroom', y='price', data=df_clean, palette='muted', hue='guestroom', legend=False)
plt.subplot(2, 3, 3)
sns.boxplot(x='basement', y='price', data=df_clean, palette='muted', hue='basement', legend=False)
plt.subplot(2, 3, 4)
sns.boxplot(x='hotwaterheating', y='price', data=df_clean, palette='muted', hue='hotwaterheating', legend=False)
plt.subplot(2, 3, 5)
sns.boxplot(x='airconditioning', y='price', data=df_clean, palette='muted', hue='airconditioning', legend=False)
plt.subplot(2, 3, 6)
sns.boxplot(x='furnishingstatus', y='price', data=df_clean, palette='muted', hue='furnishingstatus', legend=False)
plt.tight_layout()
plt.show()


### 5. Feature Encoding
We convert categorical text values into numeric variables:
- **Binary Mapping**: Yes/No columns (`mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea`) map to `1`/`0`.
- **One-Hot Encoding**: `furnishingstatus` (furnished, semi-furnished, unfurnished) maps to dummy variables with the first class dropped.



In [ ]:
# Binary mapping
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'yes': 1, 'no': 0})

# Dummy variables for furnishingstatus (using dtype=int)
df_encoded = pd.get_dummies(df_clean, columns=['furnishingstatus'], drop_first=True, dtype=int)
print("Dataset columns after encoding:")
print(df_encoded.columns.tolist())
df_encoded.head(3)


### 6. Correlation Analysis
We review correlations of all features with house prices.



In [ ]:
# Heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df_encoded.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.savefig(os.path.join("..", "Visualizations", "correlation_heatmap.png"), dpi=150, bbox_inches='tight')
plt.show()


### 7. Train-Test Split & Feature Scaling
We split data into 80% train and 20% test sets, and apply MinMax scaling to continuous predictors.



In [ ]:
# Features and target split
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

# Train-test split (random_state 100 for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=100)

print(f"Training features size: {X_train.shape}")
print(f"Testing features size: {X_test.shape}")

# Scale only continuous numerical columns to prevent leakage
scale_cols = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print("\nFirst 3 rows of scaled training features:")
print(X_train_scaled.head(3))


### 8. Model Building & Training
We fit a Multiple Linear Regression model to our scaled training data.



In [ ]:
# Initialize and train model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

print(f"Model Intercept: {lr_model.intercept_:.2f}")

# Extract coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_,
    'Abs_Coefficient': np.abs(lr_model.coef_)
}).sort_values(by='Abs_Coefficient', ascending=False)

print("\nModel Coefficients:")
print(coef_df.to_string(index=False))


### 9. Model Performance Evaluation
We evaluate the predictions on the test set using MAE, MSE, RMSE, and $R^2$ Score.



In [ ]:
# Predictions
y_pred = lr_model.predict(X_test_scaled)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"--- Evaluation Metrics (Test Set) ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R²) Score: {r2:.4f}")


### 10. Model Diagnostics
We inspect prediction spreads, error distributions (normality check), and residuals scatter (homoscedasticity check).



In [ ]:
# 1. Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='#3b82f6', edgecolors='w', s=50)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], '--r', linewidth=2, label='Perfect Fit')
plt.title('Actual vs. Predicted House Prices')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.legend()
plt.savefig(os.path.join("..", "Visualizations", "actual_vs_predicted.png"), dpi=150, bbox_inches='tight')
plt.show()

# 2. Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.7, color='#ec4899', edgecolors='w', s=50)
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.title('Residual Plot (Homoscedasticity)')
plt.xlabel('Predicted Price')
plt.ylabel('Residuals (Actual - Predicted)')
plt.savefig(os.path.join("..", "Visualizations", "residual_plot.png"), dpi=150, bbox_inches='tight')
plt.show()

# 3. Top Features Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coef_df.sort_values(by='Coefficient', ascending=False),
            palette='viridis', hue='Feature', legend=False)
plt.title('Linear Regression Coefficients (Feature Impact)')
plt.xlabel('Impact on Price')
plt.ylabel('Features')
plt.savefig(os.path.join("..", "Visualizations", "top_features.png"), dpi=150, bbox_inches='tight')
plt.show()


### 11. Key Insights & Business Recommendations

#### Key Quantitative Insights
1. **Model Fit**: The model achieves an $R^2$ score of **74.52%**, meaning that roughly 74.5% of the variance in house prices is explained by the features selected.
2. **Top Features**:
   - **Area**: Holds the highest positive impact (`+23.28 Lakhs` scaled impact). Every increase in square footage is the primary driver of property values.
   - **Bathrooms & Stories**: Have similar strong positive weights (`+14.32 Lakhs` and `+14.00 Lakhs` respectively). Adding floors or extra bathrooms provides the highest structural value.
   - **Hot Water Heating & Air Conditioning**: Adding amenities is extremely valuable, contributing `+7.61 Lakhs` and `+6.96 Lakhs` respectively to the house value.
   - **Location (prefarea)**: Preferred area location contributes `+5.25 Lakhs` to valuation, indicating location premium.
   - **Furnishing**: An unfurnished house decreases property valuation by `4.27 Lakhs` relative to a fully furnished baseline.

#### Business Recommendations
* **Homeowners & Renovators**: Upgrading structural features (such as adding a bathroom or floor) yields a much higher valuation return than cosmetic improvements. Adding modern amenities like air conditioning or central heating also offers high ROI.
* **Real Estate Agents**: Highlight property square footage (Area) and the number of bathrooms as the primary structural pillars when setting marketing guides.
* **Real Estate Investors**: Focus on properties located in "preferred areas" (prefarea) or properties with expansion potential (large area/stories) to maximize valuation arbitrage.

---
*End of Notebook. Submission-ready for Oasis Infobyte Data Analytics Internship.*

